# 01 · Caso de negocio — Venta cruzada de seguro de vehículo

Una aseguradora de salud quiere ofrecer **seguro de vehículo** a sus clientes actuales y necesita saber **a quién llamar primero**.
Ejemplo de referencia del curso: `material-del-curso/ejemplo-creditsight/notebooks/01_caso_negocio.ipynb`.

Antes de abrir un solo dato, el proyecto necesita **una pregunta de negocio, una métrica que la refleje y un costo de equivocarse**.
Este notebook termina con esas tres cosas escritas **con números**, y su resumen va a `reports/00_caso.md`.

**Tiempo en clase:** 15 minutos. **Quién lo escribe:** el rol *Negocio* del grupo, con los demás revisando.

## 1. La pregunta de negocio

**Pregunta de negocio:** ¿A cuáles de nuestros clientes de seguro de salud debemos llamar primero para ofrecerles un seguro de vehículo?

**¿Quién decide y con qué información?** El gerente de la campaña comercial decide a quién contactar y en qué orden, con el perfil del
cliente (edad, género, región, antigüedad como cliente, prima anual, canal) y de su vehículo (antigüedad, si tuvo daños, si ya tiene seguro).
El asesor hace la llamada y cierra la venta; el modelo solo ordena la lista.

**¿Qué pasa hoy, sin el modelo?** El equipo no alcanza a llamar a toda la base y elige sin una regla clara. Solo el 12,26 % de los clientes
se interesa, así que la mayoría de las llamadas no da resultado.

## 2. Equivocarse no cuesta lo mismo

**Error más caro:** no llamar a un cliente que sí estaba interesado (falso negativo). Se pierde la venta y la prima anual de la póliza.

**Error menos caro:** llamar a un cliente que no estaba interesado (falso positivo). Se pierde el tiempo de un asesor y se incomoda un poco al cliente.

**Proporción entre los dos (por ejemplo, 5 a 1) y de dónde sale:** **5 a 1, supuesto del grupo.** Una póliza vendida deja un margen
de varias veces el costo de una llamada. Debe validarse con cifras reales (margen por póliza frente a costo por llamada), y se reporta
la sensibilidad con 3 a 1 y 10 a 1.

In [ ]:
import numpy as np
import pandas as pd

# Clases del problema: Response = 0 (no interesado) y 1 (interesado).
CLASSES = ["no interesado", "interesado"]

# Matriz de costos. Filas = lo que realmente pasó, columnas = lo que decidió el modelo (no contactar / contactar).
# No contactar a un interesado cuesta 5 veces más que llamar a alguien que no lo está (supuesto del grupo).
COST_MATRIX = [[0, 1],
               [5, 0]]

if CLASSES and COST_MATRIX is not None:
    cost = pd.DataFrame(COST_MATRIX, index=[f"real {c}" for c in CLASSES],
                        columns=["no contactar", "contactar"])
    display(cost)
    # Umbral de decisión que se deduce de la matriz: contactar si P(interesado) > costo_FP / (costo_FP + costo_FN)
    print(f"Contactar si P(interesado) > {COST_MATRIX[0][1] / (COST_MATRIX[0][1] + COST_MATRIX[1][0]):.3f}")

## 3. ¿Contra qué nos comparamos? (línea base)

Una métrica sola no dice nada: necesita un punto de referencia. Calculamos el costo de **estrategias sin modelo**:
- **Siempre la clase más frecuente** (no llamar a nadie).
- **Azar según las proporciones.**
- **Llamar a todos.**
- **Una regla simple de negocio:** llamar solo a quien no tiene seguro de vehículo previo y ya tuvo daños en su vehículo.

Proporciones medidas en `dataset.csv` (381.109 filas): 87,74 % no interesado, 12,26 % interesado. El EDA las confirma.

In [ ]:
# Proporciones de cada clase, en el mismo orden que CLASSES (medidas en dataset.csv: 334.399 y 46.710 filas).
priors = np.array([334399, 46710]) / 381109

if CLASSES and COST_MATRIX is not None and priors.size == len(CLASSES):
    cm = np.array(COST_MATRIX, dtype=float)
    most_frequent = int(priors.argmax())
    always_most_frequent = priors @ cm[:, most_frequent]
    random_by_priors = priors @ cm @ priors
    call_everyone = priors @ cm[:, 1]
    print(f"Siempre '{CLASSES[most_frequent]}' (no llamar a nadie):  costo esperado = {always_most_frequent:.3f}")
    print(f"Azar según proporciones:                          costo esperado = {random_by_priors:.3f}")
    print(f"Llamar a todos:                                   costo esperado = {call_everyone:.3f}")

# Regla simple: contactar si Previously_Insured == 0 y Vehicle_Damage == 'Yes'.
# Se calcula sobre todo el dataset (descriptivo); en el EDA se recalcula solo con validación.
df = pd.read_csv("../dataset.csv")
contact = (df["Previously_Insured"] == 0) & (df["Vehicle_Damage"] == "Yes")
fp = int((contact & (df["Response"] == 0)).sum())
fn = int((~contact & (df["Response"] == 1)).sum())
tp = int((contact & (df["Response"] == 1)).sum())
rule_cost = (fp * COST_MATRIX[0][1] + fn * COST_MATRIX[1][0]) / len(df)
print(f"Regla simple:                                     costo esperado = {rule_cost:.3f}  "
      f"(contacta {contact.mean():.1%}, recall {tp / df['Response'].sum():.1%}, precisión {tp / contact.sum():.1%})")

**Lo que dice la línea base (una frase):** sin modelo, lo mejor es la regla simple (costo 0,373 por cliente frente a 0,613 de no llamar
a nadie); el modelo debe bajar de 0,373, sobre todo contactando a menos gente sin perder interesados.

**¿La exactitud (accuracy) sirve como métrica en tu caso? ¿Por qué sí o por qué no?** No. Decir siempre "no interesado" da 87,7 % de
exactitud y no encuentra a ningún cliente interesado, que es justo lo que buscamos.

## 4. Cómo se medirá el éxito

- **Métrica de negocio (principal): costo esperado por cliente** con la matriz 5 a 1 — refleja que perder un interesado cuesta 5 veces
  más que una llamada sin éxito.
- **Métrica técnica (para el torneo de modelos): PR-AUC (average precision)** — mide si los interesados quedan al principio de la lista,
  que es como se usa el modelo; con solo 12 % de positivos es más exigente que ROC-AUC. Un modelo sin información obtiene ≈ 0,123.
- **Métricas de control:** ROC-AUC; recall y precisión de "interesado" en el umbral elegido; % de interesados capturados al contactar
  al 20 %, 30 % y 50 % de la base.
- **Qué resultado haría que el negocio diga «sirve»:** costo esperado por cliente **menor que 0,373** (la regla simple) en validación,
  y confirmado una sola vez en prueba.

## 5. Lo que podría salir mal

- **Fuga de información (¿qué columna se conoce solo después de la decisión?):** `Response` se registra después del contacto. Revisar en el EDA
  que `Vintage` y `Policy_Sales_Channel` se conozcan **antes** de decidir a quién llamar. El umbral se elige en validación, nunca en prueba.
- **Sesgos o equidad (¿a quién podría perjudicar el modelo?):** podría contactar menos a ciertos grupos de edad, género o región. Se revisan
  tasa de contacto y recall por `Gender`, rangos de `Age` y `Region_Code`.
- **Datos personales o confidenciales:** no hay nombres ni documentos. `Gender` y `Age` son sensibles; `id` se excluye del modelo.
- **Hasta dónde puede decidir el modelo y dónde debe decidir una persona:** el modelo ordena la lista de llamadas; el gerente define cuántas
  llamadas hacer y el asesor decide en la conversación. El modelo mide interés declarado, no compra, y no se usa para fijar precios.

## 6. Cierre

- [x] Pregunta, quién decide y costos escritos **con números**.
- [x] Línea base calculada.
- [x] Métrica principal elegida y justificada.
- [x] Resumen copiado a `reports/00_caso.md`.
- [ ] Alguien que **no** escribió este notebook sabe explicar por qué un error cuesta más que el otro.